# Exercise Set VI - HMD


In this final set, we will look at a real actuarial example and perform a geographical analysis of mortality datafrom the Human Mortality Database (HMD). An extensive source for data on death counts, births and life tables is [mortality.org](https://mortality.org/). You will need to create an account that is for free and can then download all available data.

Note that the structure of this lab is slightly different, where we provide a significant amount of acturial context before asking you to compute something. The goal is really understanding how DS can be applied in an end-to-end manner.



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt



First, we download and import files on death counts, exposures, population and a life table for Switzerland.



In [ ]:
file = "data/lab_6/CH_deaths.txt"
data = pd.read_csv(file, skiprows = 2)
df_deaths= data[data.columns[0]].str.split(expand=True)
df_deaths.columns = data.columns.str.split(expand=True)[0]
df_deaths[df_deaths.columns.difference(['Age'])] = df_deaths[df_deaths.columns.difference(['Age'])].astype(float)


In [ ]:
file = "data/lab_6/CH_exposures.txt"
data = pd.read_csv(file, skiprows = 2)
df_exp= data[data.columns[0]].str.split(expand=True)
df_exp.columns = data.columns.str.split(expand=True)[0]
df_exp[df_exp.columns.difference(['Age'])] = df_exp[df_exp.columns.difference(['Age'])].astype(float)


In [ ]:
file = "data/lab_6/CH_population.txt"
data = pd.read_csv(file, skiprows = 2)
df_pop= data[data.columns[0]].str.split(expand=True)
df_pop.columns = data.columns.str.split(expand=True)[0]
df_pop[df_pop.columns.difference(['Age'])] = df_pop[df_pop.columns.difference(['Age'])].astype(float)


In [ ]:
file = "data/lab_6/CH_life_table.txt"
data = pd.read_csv(file, skiprows = 2)
df_life= data[data.columns[0]].str.split(expand=True)
df_life.columns = data.columns.str.split(expand=True)[0]
df_life[df_life.columns.difference(['Age'])] = df_life[df_life.columns.difference(['Age'])].astype(float)


In [ ]:
df_exp


In [ ]:
df_pop


In [ ]:
df_life



## Death probabilities

The probability of death within the next year for a person aged $x$ is 
$$ q_x = \frac{l_x - l_{x+1}}{l_x} \,,$$
where $l_x$ is the number of people alive at age $x$ and $l_{x+1}$ is the number of those people who have survived until age $x+1$. We can rewrite the formula as
$$ l_{x+1} = (1-q_x) \cdot l_x \,.$$
Additionally, we define the number of deaths
$$D_x = l_x \cdot q_x \,.$$

A first intuition at calculating `qx` might be to divide the number of deaths by the number of people alive. However, we see that the numbers do not coincide. This is due to the fact that people migrate in and out of the country.



In [ ]:
print((df_deaths["Total"]/df_pop["Total"])[0:79])
print(df_life["qx"][0:79])



We now confirm, that the central death rates `mx` are calculated as the quotient between death counts $D_x$ and exposure (~ the average number of people alive) $P_x$ for ages 0 - 79. However, from age 80 onwards, the fluctuation in these values is too strong, so a smoothing transformation has been applied.



In [ ]:
print((df_deaths["Total"]/df_exp["Total"])[0:79])
print(df_life["mx"][0:79])



We then use these central death rates to approximate the probability of death. Assuming that the people die and migrate constantly over the span of one year, we can say that the exposures (~ average number of people alive) $P_x$ is approximately

$$P_x = \frac{l_x + l_{x+1}}{2} = \frac{l_x + l_x\cdot(1-q_x)}{2} = l_x \cdot \frac{2-q_x}{2}$$

Thus the central death rate $m_x$ is
$$ m_x = \frac{D_x}{P_x} = \frac{2 q_x}{2-q_x}\,.$$

Solving this equation for $q_x$ gives
$$ q_x = \frac{m_x}{1 + 0.5 m_x}$$
and we confirm this result numerically. Note that it is slightly off for age 0, because the number of deaths is not uniformly distributed over the year for infants. Check column `ax` of the life table. This column gives the average length of survival of those who have died within that year.



In [ ]:
print(df_life["mx"]/(1+0.5*df_life["mx"]))
print(df_life["qx"])



## Life expectancy

The remaining life expectancy is defined as
$$ \begin{align}
        e_x &= \sum_{s=1}^\infty \mathbb{P}(\text{dying in year } x+s\, | \,\text{having survived until year } x) \cdot (s+a_{x+s})\\
        &= \sum_{s=1}^\infty \Big(\prod_{r = 1}^{s-1} p_{x+r}\Big) \cdot q_{x+s} \cdot (s+a_{x+s})\,.
    \end{align}$$
Note that $a_{x+s}$ is the average time to death of a person dying year $x+s$.



In [ ]:
df_life


In [ ]:
df_life.Age == "0"



### Exercise 1
Define a function `lifetime` that computes the life expectancy as a function of age `x` and calendar year `t`.



These values coincide with the last column of our data frame. Let's plot the period life expectancy at birth for newborns from 1876 to today. <br> <br>

### Exercise 2

Plot the period life expectancy of newborns as a function of the calendar year.

Let's start by filtering the data for newborns (age "0") and visualizing the trend over time.



However, this is not the true life expectancy that people experienced. Someone born in 1876 did not have the same probability to die at age 80 as someone who was 80 in 1876. He benefitted from all the medical advancements society has made until the year 1956. The true formula would look like this:
$$ e_{x,t} = \sum_{s=1}^\infty \Big(\prod_{r = 1}^{s-1} p_{x+r, t+r}\Big) \cdot q_{x+s, t+s} \cdot (s+a_{x+s, t+s})\,, $$
where the index $t$ indicates the calendar year. This would involve forecasting future mortality improvements. Let's just investigate the life expectancy for people born before 1914.<br>
We could extend the plot all the way to 2022 by assuming that there will be no more improvements after 2022.



In [ ]:
def true_lifetime(x,t):
    q = np.asarray(df_life.loc[([int(i[:3]) + t for i in df_life.Age] == df_life.Year + x) & (df_life.Year >= t)]["qx"])
    p = 1 - q
    a = np.asarray(df_life.loc[([int(i[:3]) + t for i in df_life.Age] == df_life.Year + x) & (df_life.Year >= t)]["ax"])
    e = 0
    for s in range(len(q)):
        e+= np.prod(p[:s])*q[s]*(s+a[s])
    return(e)

x = 30
year_max = 1914+x
true_expectation = []
for i in range(1876, year_max):
    true_expectation = np.append(true_expectation, true_lifetime(x,i))



Now, let's plot the period life expectancy of newborns as a function of the calendar year.


## Comparing different countries

We download and import additional lifetables for Switzerland's neighbouring countries.



In [ ]:
cols = ["Year", "Age", "ex"]
df_life = df_life[cols]
df_life.columns = ["Year", "Age", "CH"]
countries = ["CH", "FR", "IT", "DE", "AT"]
for country in countries[1:]:
    file = "data/lab_6/" + country + "_life_table.txt"
    data = pd.read_csv(file, skiprows = 2)
    temp= data[data.columns[0]].str.split(expand=True)
    temp.columns = data.columns.str.split(expand=True)[0]
    temp[temp.columns.difference(['Age'])] = temp[temp.columns.difference(['Age'])].astype(float)
    temp = temp[cols]
    temp.columns = ["Year", "Age", country]
    df_life = pd.merge(df_life, temp, how="outer", on=["Year", "Age"])


In [ ]:
df_life



### Exercise 3

Plot the development of their period life expectancies.



## Geographical plots

We would like to plot some of that information on a map. To do that, we need shapefiles that define the outlines of these countries. They are available, for example, on Eurostat [https://ec.europa.eu/eurostat/web/gisco/geodata/statistical-units/territorial-units-statistics](https://ec.europa.eu/eurostat/web/gisco/geodata/statistical-units/territorial-units-statistics).

Let's download and import these files using the library [`geopandas`](https://geopandas.org/). It is easy to plot these shapes, but without manually defining the axis limits, the plot is heavily distorted by the french overseas territories.



In [ ]:
import geopandas as gpd

fig, ax = plt.subplots(figsize=(10, 10))
#ax.set_xlim(-25, 47)
#ax.set_ylim(30, 75)

shapefile = gpd.read_file("data/lab_6/NUTS_RG_01M_2024_4326.shp")
shapefile.plot(ax = ax)
plt.show()



Taking a closer at this (geopandas-) dataframe, we see that there are different levels for the granularity of regions and all of them come with different NUTS-IDs (Nomenclature des unités territoriales statistiques) and country codes.



In [ ]:
shapefile



Now, filter the data for Switzerland.


We can select all of those countries that we are interested in through the variable `CNTR_CODE` and choose different levels of granularity on the `LEVL_CODE`. By excluding those that contain the string `FRY`, we get rid of all french overseas territories.

### Exercise 4

Select only those rows that contain information about the countries Austria, Germany, France, Italy and Switzerland. Only use regions with NUTS-Level 1 and exclude the French overseas-territories (their `NUTS_ID` starts with "FRY"). <br>
Use the function `pd.loc`. You can use an AND-operator with `&` and a NOT-operator by putting a `~` in front of a logical expression. Other helpful functions are `pd.isin()` and `pd.str.contains()`. You might need to google how to use those.




In [ ]:
#FILL OUT
#shape_file_new = shapefile.loc[ ... ]
#shape_file_new.plot("CNTR_CODE")
